# 01 — Introduction to LLMs, Mechanics, and Prompt Engineering Fundamentals

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## LLM Mechanics: Tokenization

In [3]:
text = "The quick brown fox jumps over the lazy dog."
approx_tokens = len(text) / 4
print(f"Text: {text}")
print(f"Characters: {len(text)}")
print(f"Approx tokens (BPE): {approx_tokens:.0f}")
try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    tokens = enc.encode(text)
    print(f"Actual GPT-2 BPE tokens: {len(tokens)}")
    print(f"Token IDs: {tokens}")
except ImportError:
    print("tiktoken not installed — using approximation only")


Text: The quick brown fox jumps over the lazy dog.
Characters: 44
Approx tokens (BPE): 11


Actual GPT-2 BPE tokens: 10
Token IDs: [464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290, 13]


## Anatomy of a Prompt

In [4]:
prompt_components = {
    "Instruction":    "Summarize the following article in 2 sentences.",
    "Context":        "Article: [The article text goes here...]",
    "Input":          "Artificial intelligence is transforming healthcare...",
    "Output format":  "Return: <summary>...</summary>"
}
for k, v in prompt_components.items():
    print(f"{k:20s}: {v[:60]}")


Instruction         : Summarize the following article in 2 sentences.
Context             : Article: [The article text goes here...]
Input               : Artificial intelligence is transforming healthcare...
Output format       : Return: <summary>...</summary>


## How LLMs Generate Text: Next-Token Prediction

In [5]:
prompt = "The capital of France is"
result = chat([{"role": "user", "content": prompt}], temperature=0, max_tokens=10)
print(f"Prompt : '{prompt}'")
print(f"Output : '{result.strip()}'")
print()
print("LLMs predict the most likely next token, conditioned on all prior tokens.")
print("This is autoregressive generation: each token feeds into the next prediction.")


Prompt : 'The capital of France is'
Output : 'The capital of France is Paris.'

LLMs predict the most likely next token, conditioned on all prior tokens.
This is autoregressive generation: each token feeds into the next prediction.


## Prompting vs Fine-tuning vs RAG

In [6]:
comparison = [
    ("Prompting",   "No training needed",    "Limited by context window", "Zero cost"),
    ("Fine-tuning", "Baked-in knowledge",    "Requires labeled data",     "High cost"),
    ("RAG",         "Dynamic knowledge",     "Retrieval latency",          "Medium cost"),
]
print(f"{'Approach':<15} {'Strength':<30} {'Limitation':<30} {'Cost'}")
print("-" * 85)
for row in comparison:
    print(f"{row[0]:<15} {row[1]:<30} {row[2]:<30} {row[3]}")


Approach        Strength                       Limitation                     Cost
-------------------------------------------------------------------------------------
Prompting       No training needed             Limited by context window      Zero cost
Fine-tuning     Baked-in knowledge             Requires labeled data          High cost
RAG             Dynamic knowledge              Retrieval latency              Medium cost


## Basic Prompt Quality Check

In [7]:
bad_prompt  = "Tell me about AI"
good_prompt = (
    "You are an expert AI researcher. "
    "Explain in exactly 3 bullet points what makes a large language model different "
    "from a traditional rule-based chatbot. Use plain English, no jargon."
)

print("=== BAD PROMPT ===")
print(chat([{"role": "user", "content": bad_prompt}], max_tokens=100))
print()
print("=== GOOD PROMPT ===")
print(chat([{"role": "user", "content": good_prompt}], max_tokens=150))


=== BAD PROMPT ===


Artificial Intelligence (AI) is a broad field of computer science that focuses on creating intelligent machines that can perform tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data, experiences, and interactions.
2. **Reasoning**: AI systems can make decisions, draw conclusions, and solve problems.
3. **Perception**: AI systems can interpret and understand data from sensors, such as images, speech, and text.
4. **Action**: AI systems

=== GOOD PROMPT ===


Here are three key differences between a large language model and a traditional rule-based chatbot:

• **Ability to Understand Context**: Large language models can understand the context of a conversation, including the topic, tone, and intent behind the user's words. This means they can respond more naturally and relevantly to a conversation. Traditional rule-based chatbots, on the other hand, follow a set of pre-defined rules and may not always understand the context.

• **Ability to Generate Original Responses**: Large language models can generate original responses to a user's query, rather than simply selecting from a pre-defined list of possible answers. This allows them to provide more creative and helpful responses, and to adapt to unexpected questions or topics. Traditional rule-based chatbots,
